# Encoder-router probe — OS-distill cascade labels

**Question.** Does the existing encoder-router training panel learn useful routing decisions from the OS-distill cascade relabelling of the v2-100K population?

**Dataset.** `legb_pilot/os_distill_relabel/cascade_labels.parquet` contains the cascade’s final score vector, verdict provenance, and query text for 91,093 `(dataset, query_id)` rows across 46 lanes. The cascade artifact is the only label and text source; rows with blank text are explicitly excluded because they cannot yield embeddings.

**Success criteria.** Train the existing arm panel on the text-bearing cascade rows and evaluate each saved arm on a held-out, routable lane. The deployable comparison remains the global constant; the per-lane constant remains an oracle diagnostic.

**Non-goals.** No runtime merge of v2/v3 datasets, no joint parquet, no fine-tuning BGE, and no change to arm configurations or model placement.

In [53]:
from __future__ import annotations

import os
# torch, sklearn, and lightgbm each ship their own libomp on macOS; loading two
# in one process corrupts OpenMP barriers -> SIGSEGV in the next parallel op
# (seen: kernel death inside torch.ones during EncoderRouter.fit). Must be set
# BEFORE the first import of any of them.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path

import numpy as np
import pandas as pd

from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from encoder_router.table import KEY, MODELS_DIR, ROUTES
from encoder_router.targets import OUT_DIR
from encoder_router.training import TrainingTable


def find_data_dir(start: Path) -> Path:
    """Find the repository data root from either a repo or notebook kernel cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = directory / "src" / "data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"could not find src/data above {start}")


DATA_DIR = find_data_dir(Path.cwd())
CASCADE_LABELS = DATA_DIR / "legb_pilot" / "os_distill_relabel" / "cascade_labels.parquet"
SEED = 0

print(f"cascade labels: {'ok' if CASCADE_LABELS.exists() else 'MISSING'}  {CASCADE_LABELS}")

cascade labels: ok  /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/legb_pilot/os_distill_relabel/cascade_labels.parquet


## Plan

**Metric — decisive-headroom capture (per arm × held-out lane):**

$$H = \frac{\text{captured} - \text{const}}{\text{oracle} - \text{const}}$$

The deployable bar is the global best constant computed from the training lanes. The per-lane best constant is retained only as an oracle diagnostic: it assumes a lane label at serving time. Rows where the route scores do not differ carry no dense-vs-sparse routing signal and are excluded by the existing decisive-row evaluation.

**Probe arms — unchanged.** Every variant retains `corpus_branch=False` except `design_branches`, so the panel continues to separate the base MLP, shuffled-target floor, taxonomy inputs, Zipf inputs, LightGBM, and the existing design branch. Each arm is fit once with `FAIR_LANE` held out and saved in the established model directory.

## 1. Load cascade labels and validate query text

The cascade artifact is the sole source of scores, verdict provenance, and query text. IDs are normalized to strings for the downstream embedding cache and catalog joins. Rows with absent or blank query text are reported and excluded explicitly.

In [54]:
cascade_raw = pd.read_parquet(CASCADE_LABELS).copy()
cascade_raw["query_id"] = cascade_raw["query_id"].astype(str)

required = {"dataset", "query_id", "label_layer", *(f"score_{route}" for route in ROUTES)}
missing_columns = required.difference(cascade_raw.columns)
assert not missing_columns, f"cascade labels missing columns: {sorted(missing_columns)}"
assert not cascade_raw.duplicated(KEY).any(), "cascade labels must be unique on (dataset, query_id)"

print(f"cascade rows: {len(cascade_raw):,} across {cascade_raw['dataset'].nunique()} lanes")
display(cascade_raw["label_layer"].value_counts().rename_axis("label_layer").to_frame("rows"))

cascade rows: 91,093 across 46 lanes


,rows
label_layer,
l1,42871
judge_queue_tied,17252
argmax_l1_l2,8132
unmeasured_tied,8098
judge_queue_zero,6728
depth100_fallback,5044
unmeasured_zero,2968


In [55]:
has_text = cascade_raw["query"].notna() & cascade_raw["query"].astype(str).str.strip().ne("")
text_missing = cascade_raw.loc[~has_text].groupby("dataset").size().sort_values(ascending=False)

print(f"query text available: {int(has_text.sum()):,}/{len(cascade_raw):,} rows")
print(f"excluded (blank query text): {int((~has_text).sum()):,} rows")
display(text_missing.rename("rows_without_text").to_frame())

cascade_trainable = cascade_raw.loc[has_text].reset_index(drop=True)
assert cascade_trainable["query"].notna().all()
assert cascade_trainable["query"].astype(str).str.strip().ne("").all()

query text available: 91,080/91,093 rows
excluded (blank query text): 13 rows


,rows_without_text
dataset,
crumb-code-retrieval,13


## 2. Build the training table from the cascade labels

`AcceptabilityLabels` derives the existing `ok_*` targets, the cost-aware serving decision, and the outcome shape from the cascade score vector — nothing here depends on the compact cascade schema carrying a `shape` column.

In [56]:
USE_CLEAN_SCORES = True   # cascade rows: train on argmax(l1,l2) with l1 fallback -> the top@100
# fabrication reverts to all_zero (kept, not trained). Needs the component score columns from
# os_distill_argmax.ipynb; falls back to the stored final score when they are absent.

def apply_score_policy(df, use_clean=USE_CLEAN_SCORES):
    """Clean cascade score = argmax(l1,l2).fillna(l1); depth-100 rows -> all_zero."""
    if not use_clean:
        return df
    if not all(f"score_{r}__argmax" in df.columns and f"score_{r}__l1" in df.columns for r in ROUTES):
        print("component score columns absent — re-run os_distill_argmax.ipynb assembly; using final scores")
        return df
    df = df.copy()
    for r in ROUTES:
        df[f"score_{r}"] = df[f"score_{r}__argmax"].fillna(df[f"score_{r}__l1"])
    print("score policy: CLEAN = argmax(l1,l2).fillna(l1) — top@100 noise dropped")
    return df

# Add the previous datasets alongside the cascade. cascade (the v2-100K relabel) WINS key
# collisions; the extra sources contribute only their most-decisive rows via SOURCE_MODE:
# "all" keeps every decisive row, "margin" keeps only margin>=MARGIN (sub-margin -> all_tied,
# dropped by a decisive_only arm). So v2/v3 add clean signal without their tie-noise.
EXTRA_SOURCES = {
    "v2":     DATA_DIR / "route_labels" / "labels.parquet",
    "v3":     DATA_DIR / "v3" / "labels.parquet",
    "v3_aug": DATA_DIR / "v3" / "augmented" / "labels.parquet",
}
MARGIN      = 0.4
SOURCE_MODE = {"cascade": "all", "v2": "margin", "v3": "margin", "v3_aug": "margin"}
CANON = list(KEY) + ["query", *(f"score_{r}" for r in ROUTES)]

def build_union(cascade_trainable):
    """Clean cascade + text-bearing rows from EXTRA_SOURCES; cascade wins on key collision."""
    clean = apply_score_policy(cascade_trainable)[CANON + ["label_layer"]].assign(_source="cascade")
    parts = []
    for tag, path in EXTRA_SOURCES.items():
        df = pd.read_parquet(path).astype({"query_id": str})
        df = df[df["query"].notna() & df["query"].astype(str).str.strip().ne("")]
        parts.append(df[CANON].assign(_source=tag, label_layer=tag))
    parts.append(clean)   # cascade LAST -> wins keep="last"
    union = pd.concat(parts, ignore_index=True).drop_duplicates(list(KEY), keep="last")
    return union.reset_index(drop=True)

def margin_gate(frame):
    """Per-source: 'margin' sources keep only rows whose top route beats the runner-up by
    >= MARGIN; sub-margin rows -> all_tied (dropped by a decisive_only arm, else tie_weight)."""
    sc = np.sort(frame[[f"score_{r}" for r in ROUTES]].to_numpy(float), axis=1)[:, ::-1]
    margin = sc[:, 0] - sc[:, 1]
    mode = frame["_source"].map(lambda s: SOURCE_MODE.get(s, "all")).to_numpy()
    demote = (mode == "margin") & (frame["shape"].to_numpy() == "routes_differ") & (margin < MARGIN)
    if demote.any():
        frame = frame.copy()
        frame.loc[demote, "shape"] = "all_tied"
        print(f"  margin gate: demoted {int(demote.sum()):,} sub-{MARGIN} rows -> all_tied "
              f"from {sorted(frame.loc[demote, '_source'].unique())}")
    return frame

def build_table(labels):
    """AcceptabilityLabels over the canonical score triple, then the per-source margin gate."""
    merged = margin_gate(AcceptabilityLabels(labels, tolerance=None).frame())
    table = TrainingTable()
    table.__dict__["frame"] = merged.reset_index(drop=True)
    return table

union = build_union(cascade_trainable)
table = build_table(union)
print(f"union {len(union):,} rows | by source {union['_source'].value_counts().to_dict()}")
print(f"decisive (routes_differ) after gate: {int((table.frame['shape'] == 'routes_differ').sum()):,}")
print(table.frame["shape"].value_counts().to_string())

score policy: CLEAN = argmax(l1,l2).fillna(l1) — top@100 noise dropped
  margin gate: demoted 51,383 sub-0.4 rows -> all_tied from ['v2', 'v3', 'v3_aug']
union 233,247 rows | by source {'v3_aug': 91280, 'cascade': 91080, 'v3': 35005, 'v2': 15882}
decisive (routes_differ) after gate: 66,080
shape
all_tied         120414
routes_differ     66080
all_zero          46753


In [57]:
# The compact cascade artifact is intentionally score-first. Verify the reconstructed
# training frame has the inputs and targets the downstream arms require.
required_training_columns = {"query", "shape", "serve", *(f"ok_{route}" for route in ROUTES)}
missing_training_columns = required_training_columns.difference(table.frame.columns)
assert not missing_training_columns, f"training frame missing: {sorted(missing_training_columns)}"
assert not table.frame.duplicated(KEY).any()
table.frame[KEY + ["query", "label_layer", "shape", "serve"]].head()

,dataset,query_id,query,label_layer,shape,serve
0,scirgen-geo-en,db95e56d-37af-49d9-9e2c-f654067bcc3d-Feature Specification-0,What are the characteristics of the data collection method...,v2,all_tied,sparse_only
1,scirgen-geo-en,f5f4b665-e5cc-4c1a-94c9-7675c6298d42-Instrumental/Procedur...,What procedures and instruments are used to measure and ve...,v2,routes_differ,pure_rrf
2,scirgen-geo-en,93a15551-640e-40d4-a247-7e140b92b3a7-Causal Consequence-0,What are the potential consequences on data precision and ...,v2,all_tied,sparse_only
3,scirgen-geo-en,3e1948b4-ec66-443e-a57d-6ce42747afcc-Quantification-0,What is the predicted population size for each future year...,v2,all_tied,sparse_only
4,scirgen-geo-en,8d2a4a4d-1475-40eb-a553-4f489b2b75ad-Quantification-0,What is the spatial resolution of a snow cover dataset rel...,v2,all_tied,sparse_only


## 3. Probe arms — 6 configurations

Six arm configurations are trained once and saved in §3b: `no_branches`, `shuffled_targets`, `features_input_nocorpus`, `zipf_input_nocorpus`, `lightgbm_nocorpus`, and `design_branches`. Their inputs, branches, learners, and held-out evaluation protocol are unchanged from the source notebook.

In [64]:
from IPython.display import display
from encoder_router.evaluate import Arm, E5, GEMINI

PROBE_ARMS = (
    Arm("no_branches",             cell_branch=False, corpus_branch=False),
    Arm("shuffled_targets",        shuffle_targets=True, corpus_branch=False),
    Arm("features_input_nocorpus", feature_inputs=True, cell_branch=False, corpus_branch=False),
    Arm("zipf_input_nocorpus",     zipf_inputs=True,    cell_branch=False, corpus_branch=False),
    Arm("zipf_shape_nocorpus",     zipf_inputs=True,    shape_inputs=True, cell_branch=False, corpus_branch=False),
    Arm("bge_plus_e5_nocorpus",    aux_embedding_model=E5, cell_branch=False, corpus_branch=False),  # 2-encoder; gemini plugs same slot once QueryEmbeddings speaks it
    Arm("bge_plus_e5_zipf",    aux_embedding_model=E5, cell_branch=False, corpus_branch=False, zipf_inputs=True, shape_inputs=True),  # 2-encoder; gemini plugs same slot once QueryEmbeddings speaks it
    Arm("lightgbm_nocorpus",       feature_inputs=True, learner="lgbm",
                                   cell_branch=False, corpus_branch=False),
    # LUPI: predict corpus/gold-doc profiles from the query latent (a "hidden corpus
    # representation") and feed it forward to the route layers — serve-safe (no corpus at
    # inference). The one arm that can carry collection-relative signal query-only.
    Arm("design_branches",         cell_branch=True, corpus_branch=True),
)

import os  # paid gemini complement joins the sweep ONLY when the spend gate is on,
# so a gate-off run trains bge/e5/shape/etc without train_arm raising on gemini.
# if os.environ.get("ROUTER_EMBED_LIVE") == "1":
#     PROBE_ARMS = PROBE_ARMS + (
#         Arm("bge_plus_gemini_nocorpus", aux_embedding_model=GEMINI,
#             cell_branch=False, corpus_branch=False),
#     )
#     print("ROUTER_EMBED_LIVE=1 -> bge_plus_gemini_nocorpus added (paid gemini embeds)")

def _with_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Derive headroom over the per-lane BEST constant (+ served_max) for the eye tests."""
    for col in ("train_val_loss", "train_epochs"):
        if col not in df.columns:
            df = df.assign(**{col: np.nan})
    const_cols = [c for c in ("const_dense", "const_sparse", "const_rrf") if c in df.columns]
    best_const = df[const_cols].max(axis=1)
    served_cols = [c for c in ("served_dense_only", "served_sparse_only", "served_pure_rrf") if c in df.columns]
    served_max = df[served_cols].max(axis=1)
    gap = (df["oracle"] - best_const).replace(0, np.nan)
    return df.assign(
        best_const   = best_const,
        served_max   = served_max,
        headroom     = (df["captured"] - best_const) / gap,
        oracle_ratio = df["captured"] / df["oracle"].replace(0, np.nan),
    )

## 3a. Pick the validation lane FIRST — held out during training (no leakage)

Rank the cascade’s available lanes by routable headroom and route diversity, select `FAIR_LANE`, and exclude it from §3b fitting. The saved arm records the holdout in `meta.json`; §4e evaluates that exact artifact on the held-out lane.

In [65]:
# 4e-pre — rank lanes by routable headroom (oracle - best_const) and route diversity
MIN_GAP = 0.10             # routable headroom over the best constant a lane must offer
MAX_DOMINANT_SHARE = 0.55  # truth routes must be balanced: no single route may exceed this share
MIN_ROWS = 300             # enough decisive rows for a stable held-out estimate

def rank_lanes(table, min_rows=200):
    dec = table.frame[(table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    rows = []
    for lane, g in dec.groupby("dataset"):
        if len(g) < min_rows:
            continue
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        consts = {r: float(sc[r].mean()) for r in ROUTES}
        best_const = max(consts.values())
        oracle = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        mix = g["serve"].value_counts(normalize=True)
        rows.append({"lane": lane, "n": len(g), "best_const": best_const, "oracle": oracle,
                     "routable_gap": oracle - best_const,
                     "best_route": max(consts, key=consts.get),
                     "dominant_serve": mix.idxmax(), "dominant_share": float(mix.max())})
    return pd.DataFrame(rows).sort_values("routable_gap", ascending=False).reset_index(drop=True)

ranked = rank_lanes(table)
print("lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):")
display(ranked.round(3).head(15))
print("clerc (the bad default) for contrast:")
display(ranked[ranked["lane"] == "clerc"].round(3))

_fair = ranked[(ranked["routable_gap"] > MIN_GAP)
               & (ranked["dominant_share"] < MAX_DOMINANT_SHARE)
               & (ranked["n"] >= MIN_ROWS)]
FAIR_LANE = _fair.iloc[0]["lane"] if len(_fair) else ranked.iloc[0]["lane"]
print(f"\nfair candidates (gap>{MIN_GAP}, dominant_share<{MAX_DOMINANT_SHARE}, n>={MIN_ROWS}): "
      f"{_fair['lane'].tolist()[:8]}")
print(f"-> FAIR_LANE = {FAIR_LANE!r}  (used by §3b/§4e; override freely, or loop over _fair['lane'])")

lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
0,crumb-clinical-trial,1031,0.572,0.978,0.406,sparse_only,sparse_only,0.541
1,bright-earth-science,227,0.540,0.870,0.330,dense_only,sparse_only,0.511
2,bright-biology,419,0.627,0.916,0.288,dense_only,dense_only,0.556
3,freshstack-laravel,1140,0.662,0.948,0.286,dense_only,dense_only,0.608
4,trec-dl-2022,235,0.604,0.864,0.260,sparse_only,sparse_only,0.711
5,freshstack-langchain,231,0.380,0.639,0.258,dense_only,sparse_only,0.667
6,rarb-math,4304,0.483,0.738,0.255,pure_rrf,sparse_only,0.584
7,freshstack-yolo,402,0.710,0.959,0.250,dense_only,dense_only,0.679
8,scirgen-geo-en,10342,0.393,0.607,0.215,sparse_only,sparse_only,0.780
9,techqa,209,0.545,0.754,0.209,pure_rrf,sparse_only,0.612


clerc (the bad default) for contrast:


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
11,clerc,3052,0.676,0.857,0.181,sparse_only,sparse_only,0.794



fair candidates (gap>0.1, dominant_share<0.55, n>=300): ['crumb-clinical-trial', 'orcas', 'gooaq']
-> FAIR_LANE = 'crumb-clinical-trial'  (used by §3b/§4e; override freely, or loop over _fair['lane'])


## 3b. Train & save ONE model per arm — holding out FAIR_LANE

Per arm, fit ONE model on all cascade-training rows except `FAIR_LANE` and save to `BASE/<arm.name>/`. The held-out lane is recorded in `meta.json`, so §4e validates the exact saved model with no leakage. `FORCE_RETRAIN=True` overwrites; re-runs otherwise skip matching cached arms.

In [66]:
# 3b — per-arm: train ONE model per arm on all lanes EXCEPT FAIR_LANE (held out), save each
import json as _json, joblib, time as _time
from pathlib import Path as _P
from encoder_router.evaluate import LaneCV, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import (EMBEDDING_PREFIXES, HEAD_ROUTES, LexicalShape, NgramSvd,
                                   ZipfStats, serve_from_probabilities)
from encoder_router.training import QueryEmbeddings

TRAIN_VERSION = f"2026-09-09-cascade-union-m{MARGIN}-{sorted(SOURCE_MODE.items())}"   # union+per-source margin -> config invalidates cache

def _zstats(a):
    m = a.mean(0); s = a.std(0); s[s == 0] = 1.0
    return m.astype(np.float32), s.astype(np.float32)

def train_arm(table, arm, base_dir, holdout_lane=None, seed=SEED,
              pool_mask=None, train_version=None, split_mode="lane_holdout"):
    """Train ONE model faithful to `arm` on all lanes EXCEPT holdout_lane; save to base_dir/arm.name."""
    t0 = _time.perf_counter()
    def log(msg): print(f"  [{arm.name}] +{_time.perf_counter()-t0:6.1f}s  {msg}", flush=True)

    frame = table.frame
    if pool_mask is None:
        pool_mask = np.ones(len(frame), bool) if holdout_lane is None else (frame["dataset"] != holdout_lane).to_numpy()
    pool = np.flatnonzero(pool_mask)
    log(f"START learner={arm.learner} feature_inputs={arm.feature_inputs} zipf_inputs={arm.zipf_inputs}")
    log(f"holdout_lane={holdout_lane!r} | train on {len(pool):,}/{len(frame):,} rows "
        f"({frame.loc[pool_mask, 'dataset'].nunique()} lanes)")
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame); log(f"embeddings {emb.shape}")
    if arm.aux_embedding_model:
        aux = QueryEmbeddings(arm.aux_embedding_model).matrix(frame)
        emb = np.concatenate([emb, aux], axis=1); log(f"aux encoder {arm.aux_embedding_model} -> {emb.shape}")
    svd = NgramSvd(seed=seed).fit(frame.loc[pool_mask, "query"]); log("ngram-SVD fit (train lanes only)")
    blocks, stats = [emb, svd.transform(frame["query"])], {}
    if arm.feature_inputs:
        log("assembling taxonomy feature inputs...")
        f = table.feature_matrix.to_numpy(np.float32); m, s = _zstats(f[pool_mask])
        stats["feat"] = (m, s); blocks.append((f - m) / s); log(f"feature inputs {f.shape}")
    if arm.zipf_inputs:
        z = ZipfStats().frame(frame["query"]).to_numpy(np.float32); m, s = _zstats(z[pool_mask])
        stats["zipf"] = (m, s); blocks.append((z - m) / s); log(f"zipf inputs {z.shape}")
    if arm.shape_inputs:
        sh = LexicalShape().frame(frame["query"]).to_numpy(np.float32); m, s = _zstats(sh[pool_mask])
        stats["shape"] = (m, s); blocks.append((sh - m) / s); log(f"shape inputs {sh.shape}")
    x = np.concatenate(blocks, axis=1).astype(np.float32)
    route = table.route_targets().to_numpy(np.float32)
    log(f"input matrix x={x.shape}")
    cell_t, corpus_t, feat_t = LaneCV(table, seed=seed)._targets(arm, pool_mask)
    log(f"branches: cell={None if cell_t is None else cell_t.shape} "
        f"corpus={None if corpus_t is None else corpus_t.shape} feature={None if feat_t is None else feat_t.shape}")
    if arm.shuffle_targets:                     # real permutation FLOOR: shuffle ROUTE labels in-pool
        rp = np.random.default_rng(seed + 1).permutation(len(pool))
        route = route.copy(); route[pool] = route[pool][rp]
        log("shuffled ROUTE labels within train pool (permutation floor; thresholds still use real labels)")

    p = _P(base_dir) / arm.name; p.mkdir(parents=True, exist_ok=True)
    if arm.learner == "lgbm":
        from lightgbm import LGBMClassifier
        models = []
        for i, head in enumerate(HEAD_ROUTES):
            ok = (~np.isnan(route[:, i])) & pool_mask; pos = route[ok, i].sum()
            log(f"lgbm head '{head}': fit on {int(ok.sum()):,} rows ({int(pos):,} positive)")
            models.append(LGBMClassifier(n_estimators=400, learning_rate=0.05, n_jobs=1,
                random_state=seed, verbose=-1,
                scale_pos_weight=float(np.clip((ok.sum() - pos) / max(pos, 1), 1, 100))
                ).fit(x[ok], route[ok, i]))
        joblib.dump(models, p / "lgbm.joblib"); log("lgbm models saved")
        probs_tr = pd.DataFrame(np.column_stack([m.predict_proba(x[pool])[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        rng = np.random.default_rng(seed); perm = rng.permutation(len(pool)); cut = max(len(pool) // 10, 1)
        fi, vi = pool[perm[cut:]], pool[perm[:cut]]
        log(f"mlp fit: {len(fi):,} train / {len(vi):,} val  (fit bar tracks epochs)")
        er = EncoderRouter(seed=seed).fit(x[fi], route[fi],
            None if cell_t is None else cell_t[fi], None if corpus_t is None else corpus_t[fi],
            None if feat_t is None else feat_t[fi],
            x_val=x[vi], val_route_targets=route[vi])
        log(f"mlp done: {len(er.history)} epochs, best_val={getattr(er, 'best_val_loss', float('nan')):.4f}")
        er.save(p / "router.pt"); log("router.pt saved"); probs_tr = er.probabilities(x[pool])
    thr = tuned_thresholds(probs_tr, frame.loc[pool_mask]); log(f"tuned thresholds {np.round(thr, 3).tolist()}")
    svd.save(p / "svd.joblib"); np.save(p / "thresholds.npy", thr)
    for k, (m, s) in stats.items():
        np.save(p / f"{k}_mean.npy", m); np.save(p / f"{k}_std.npy", s)
    mix = pd.Series(serve_from_probabilities(probs_tr, thr)).value_counts(normalize=True).round(2).to_dict()
    log(f"served mix (train lanes): {mix}")
    (p / "meta.json").write_text(_json.dumps(
        {"arm": arm.name, "learner": arm.learner, "embedding_model": arm.embedding_model,
         "prefix": EMBEDDING_PREFIXES.get(arm.embedding_model, ""),
         "feature_inputs": arm.feature_inputs, "zipf_inputs": arm.zipf_inputs,
          "shape_inputs": arm.shape_inputs, "aux_embedding_model": arm.aux_embedding_model,
         "serve_safe": not arm.feature_inputs, "holdout_lane": holdout_lane,
         "split_mode": split_mode,
         "trained_at": _time.strftime("%Y-%m-%d %H:%M:%S"), "train_version": train_version or TRAIN_VERSION}))
    log(f"SAVED -> {p}  (held out {holdout_lane!r}, total {_time.perf_counter()-t0:.1f}s)")
    return p

BASE = MODELS_DIR / "os_distill_relabel"     # ISOLATED experimental dir — never write cascade models into the serving/DVC classifiers_union_200k
FORCE_RETRAIN = False                          # True -> retrain & OVERWRITE every arm, even if saved

def _cache_ok(arm, base, holdout):
    """A saved arm is reusable ONLY if its manifest matches this run's holdout lane + config.
    A bare exists() check would silently reuse pre-holdout (all-lanes, leaked) models."""
    p = base / arm.name; mp = p / "meta.json"
    if not mp.exists():
        return False
    try:
        m = _json.loads(mp.read_text())
    except Exception:
        return False
    weights = "lgbm.joblib" if arm.learner == "lgbm" else "router.pt"
    return (m.get("train_version") == TRAIN_VERSION and m.get("holdout_lane") == holdout
            and m.get("arm") == arm.name and m.get("learner") == arm.learner
            and m.get("feature_inputs") == arm.feature_inputs
            and m.get("zipf_inputs") == arm.zipf_inputs
            and m.get("shape_inputs", False) == arm.shape_inputs
            and m.get("aux_embedding_model") == arm.aux_embedding_model and (p / weights).exists())

_todo   = list(PROBE_ARMS) if FORCE_RETRAIN else [a for a in PROBE_ARMS if not _cache_ok(a, BASE, FAIR_LANE)]
_cached = [] if FORCE_RETRAIN else [a.name for a in PROBE_ARMS if _cache_ok(a, BASE, FAIR_LANE)]
print(f"per-arm training | holdout={FAIR_LANE!r} | cascade table {len(table.frame):,} rows | FORCE_RETRAIN={FORCE_RETRAIN}")
print(f"  cached (skip): {_cached or '-'}")
print(f"  to train:      {[a.name for a in _todo] or '-'}", flush=True)
for _n, _a in enumerate(_todo, 1):
    print(f"===== arm {_n}/{len(_todo)}: {_a.name} =====", flush=True)
    train_arm(table, _a, BASE, holdout_lane=FAIR_LANE)
    print(flush=True)
print("all arms saved." if _todo else "nothing to train - all arms cached.")

per-arm training | holdout='crumb-clinical-trial' | cascade table 233,247 rows | FORCE_RETRAIN=False
  cached (skip): ['no_branches', 'shuffled_targets', 'features_input_nocorpus', 'zipf_input_nocorpus', 'zipf_shape_nocorpus', 'bge_plus_e5_nocorpus', 'bge_plus_e5_zipf', 'lightgbm_nocorpus', 'design_branches']
  to train:      -
nothing to train - all arms cached.


## 4b. Eye test — inspect the cascade lanes and real query errors

First inspect the available training population by lane and label layer. Then refit one leave-one-lane-out fold for a selected lane and read the decisive rows where the router disagrees with the score-derived serving route. There is no baseline-vs-union delta here: this notebook has one label source only.

In [67]:
# EYE TEST A — cascade coverage by lane and label layer
lane_population = (
    table.frame.groupby("dataset")
    .agg(
        rows=("query_id", "size"),
        decisive=("shape", lambda s: int((s == "routes_differ").sum())),
        labelled=("serve", lambda s: int(s.notna().sum())),
    )
    .assign(decisive_share=lambda d: d["decisive"] / d["rows"])
    .sort_values(["decisive", "rows"], ascending=False)
)
display(lane_population.head(20).round(3))
print("label-layer mix among trainable cascade rows:")
display(table.frame["label_layer"].value_counts().rename("rows").to_frame())

,rows,decisive,labelled,decisive_share
dataset,,,,
quest,47287,13243,44898,0.280
scirgen-geo-en,61797,10342,33663,0.167
crumb-legal-qa,8000,5948,6881,0.744
rarb-math,7912,4304,7777,0.544
finder,5703,3581,3993,0.628
crumb-code-retrieval,4354,3566,3917,0.819
gooaq,7086,3114,6816,0.439
clerc,15190,3052,13955,0.201
orcas,7618,2663,7016,0.350


label-layer mix among trainable cascade rows:


,rows
label_layer,
v3_aug,91280
l1,42871
v3,35005
judge_queue_tied,17252
v2,15882
argmax_l1_l2,8132
unmeasured_tied,8098
judge_queue_zero,6728
depth100_fallback,5044


In [68]:
# EYE TEST B — read the real queries the router gets wrong on a chosen lane (one refit)
from encoder_router.evaluate import LaneCV
from encoder_router.training import QueryEmbeddings

def query_drilldown(table, lane, arm=PROBE_ARMS[0], seed=SEED):
    """Refit one LOO fold and return per-query rows with truth vs router serve."""
    cv = LaneCV(table, seed=seed)
    frame = table.frame
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame)
    train = (frame["dataset"] != lane).to_numpy()
    x = cv._inputs(arm, frame, emb, train)
    route = table.route_targets().to_numpy(dtype=np.float32)
    served, _thr, _fit = cv._served(arm, x, route, train, frame)
    test = frame[~train][["dataset", "query_id", "query", "shape", "serve",
                          "score_dense_only", "score_sparse_only", "score_pure_rrf"]].copy()
    test["served"] = served
    return test

LANE = FAIR_LANE                    # <- override with a lane from EYE TEST A
dd = query_drilldown(table, LANE)

differ = dd[(dd["shape"] == "routes_differ") & dd["serve"].notna()]
wrong = differ[differ["served"] != differ["serve"]]
print(f"lane={LANE} (cascade): "
      f"{len(dd)} test rows | {len(differ)} decisive | router wrong on {len(wrong)} "
      f"({len(wrong)/max(len(differ),1):.0%})")
print(f"served mix: {dd['served'].value_counts(normalize=True).round(2).to_dict()}  "
      f"vs truth: {differ['serve'].value_counts(normalize=True).round(2).to_dict()}")
pd.set_option("display.max_colwidth", 90)
print("\ndecisive rows where router != truth (read the queries):")
display(wrong[["query", "serve", "served",
               "score_dense_only", "score_sparse_only", "score_pure_rrf"]].head(20))

fit:   0%|          | 0/200 [00:00<?, ?it/s]

lane=crumb-clinical-trial (cascade): 8776 test rows | 1031 decisive | router wrong on 454 (44%)
served mix: {'sparse_only': 0.95, 'dense_only': 0.05}  vs truth: {'sparse_only': 0.54, 'dense_only': 0.42, 'pure_rrf': 0.04}

decisive rows where router != truth (read the queries):


,query,serve,served,score_dense_only,score_sparse_only,score_pure_rrf
126781,mouthwash for reducing inflammation after dental implant surgery,sparse_only,dense_only,0.094639,1.000000,0.189279
126785,MCI-186 pharmacokinetics hepatic impairment single dose study,dense_only,sparse_only,1.000000,0.189279,0.189279
126786,vitamin D 50000 IU supplementation safety efficacy deficiency treatment,pure_rrf,sparse_only,0.189279,0.129203,1.000000
126798,hyperthermic intraperitoneal chemotherapy peritoneal cancer treatment,dense_only,sparse_only,1.000000,0.094639,0.189279
126824,testosterone treatment chronic heart failure clinical trial,dense_only,sparse_only,1.000000,0.189279,0.189279
126880,weight loss program for sleep apnea treatment,dense_only,sparse_only,1.000000,0.000000,0.150000
126907,COVID-19 antibody response hemodialysis patients versus non-dialysis patients,dense_only,sparse_only,1.000000,0.189279,0.189279
126971,effectiveness of western medicine versus traditional chinese medicine primary care,dense_only,sparse_only,1.000000,0.189279,0.189279
126979,left bundle branch pacing vs biventricular pacing heart failure,dense_only,sparse_only,1.000000,0.189279,0.189279
126987,sirolimus eluting stent versus bare metal stent diabetic coronary artery disease,dense_only,sparse_only,1.000000,0.000000,0.189279


## 4c. Load a saved arm & classify (no training here — models come from §3b)

`load_classifier(BASE/"<arm>")` reloads one arm's exact saved weights into `classify(queries)`.
Serve-safe arms (`no_branches` / `shuffled_targets` / `zipf_input`) classify raw strings;
`features_input` / `lightgbm` are saved but need the taxonomy extractor at serve time
(`serve_safe=False`), so raw-query classify is blocked. Serving policy stays gate + constant.

In [50]:
# 4c — load a saved arm; serve by tuned THRESHOLDS, then hedge close calls to pure_rrf
from sentence_transformers import SentenceTransformer
from encoder_router.table import LexicalShape
from encoder_router.training import QueryEmbeddings   # aux-encoder path (e5/gemini)

RRF_DELTA = 0.07   # override to pure_rrf when |p_dense - p_sparse| < RRF_DELTA (hedge on top of thresholds)

def serve_thr_rrf(probs, thr, delta=RRF_DELTA):
    """Tuned-threshold serving (unchanged), with a pure_rrf hedge overlaid on near-ties."""
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); s = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - s) < delta, "pure_rrf", base)

def load_classifier(arm_dir):
    """Reload ONE arm's saved model into classify(queries) — no retraining. Builds the
    SAME input stack training used: [primary emb; aux emb; svd; zipf; shape]."""
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    st = SentenceTransformer(meta["embedding_model"]); prefix = meta["prefix"]
    aux_model = meta.get("aux_embedding_model")     # e.g. gemini — hosted, needs ROUTER_EMBED_LIVE=1
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities
    zstat = (np.load(p / "zipf_mean.npy"), np.load(p / "zipf_std.npy")) if meta["zipf_inputs"] else None
    sstat = (np.load(p / "shape_mean.npy"), np.load(p / "shape_std.npy")) if meta.get("shape_inputs") else None

    def classify(queries, delta=RRF_DELTA):
        if not meta["serve_safe"]:
            raise RuntimeError(f"{meta['arm']} uses taxonomy feature_inputs — assemble features first")
        q = [queries] if isinstance(queries, str) else list(queries)
        emb = np.asarray(st.encode([prefix + t for t in q], normalize_embeddings=True))
        if aux_model:                                # concat the complementary encoder (order matches training)
            emb = np.concatenate([emb, QueryEmbeddings(aux_model)._encode(q, 100)], axis=1)
        blocks = [emb, svd.transform(pd.Series(q))]
        if zstat is not None:
            z = ZipfStats().frame(pd.Series(q)).to_numpy(np.float32)
            blocks.append((z - zstat[0]) / zstat[1])
        if sstat is not None:
            sh = LexicalShape().frame(pd.Series(q)).to_numpy(np.float32)
            blocks.append((sh - sstat[0]) / sstat[1])
        probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
        return probs.assign(route=serve_thr_rrf(probs, thr, delta), query=q)
    return classify

# classify = load_classifier(BASE / "bge_plus_e5_nocorpus")   # reload any arm by name
classify = load_classifier(BASE / "no_branches")
pd.set_option("display.max_colwidth", 70)
display(classify([
    "how do I refresh laravel migrations programmatically",
    "United States v. Fumo evidence Pennsylvania Ethics Act admissibility",
    "what is the difference between computer engineering and computer science",
]))


,sparse_only,dense_only,route,query
0,0.414861,0.659300,sparse_only,how do I refresh laravel migrations programmatically
1,0.697093,0.282055,sparse_only,United States v. Fumo evidence Pennsylvania Ethics Act admissibility
2,0.487126,0.753002,dense_only,what is the difference between computer engineering and computer s...


## 4d. Diagnostic battery — probe the dense/sparse decision boundary

Curated cases grouped by the mechanism they stress. `expect` is a mechanism-based HEURISTIC
(rare identifiers/codes → sparse; conceptual NL → dense; concept+rare-term → hybrid), NOT ground
truth — the interesting rows are the DIVERGENCES. The load-bearing category is `buried-id`: a rare
discriminating token inside fluent text. If the router serves dense there, it is reading fluency
and missing the rare token — the router-rarity-gap made visible on queries you control.

## 2. Build the training table from the cascade labels

`AcceptabilityLabels` derives the existing `ok_*` targets, the cost-aware serving decision, and the outcome shape from the cascade score vector — nothing here depends on the compact cascade artifact carrying a `shape` column.

In [51]:
# 4d — diagnostic battery: does the router use lexical rarity, or default to dense on fluent text?
CASES = [
    ("id/code",   "sparse_only", "CVE-2021-44228 log4j remote code execution"),
    ("id/code",   "sparse_only", "ORA-00942 table or view does not exist"),
    ("id/code",   "sparse_only", "kubectl pod CrashLoopBackOff exit code 137"),
    ("id/code",   "sparse_only", "pip ImportError libGL.so.1 cannot open shared object file"),
    ("id/code",   "sparse_only", "doi:10.1038/s41586-020-2649-2"),
    ("buried-id", "sparse_only", "why does my postgres connection fail with ECONNREFUSED 127.0.0.1:5432"),
    ("buried-id", "sparse_only", "what causes the E11000 duplicate key error in mongodb"),
    ("buried-id", "sparse_only", "how do I fix Objects are not valid as a React child"),
    ("concept",   "dense_only",  "how does photosynthesis differ from cellular respiration"),
    ("concept",   "dense_only",  "what are the ethical implications of autonomous weapons"),
    ("concept",   "dense_only",  "explain the difference between empathy and sympathy"),
    ("concept",   "dense_only",  "why do people procrastinate even when they know the costs"),
    ("mixed",     "pure_rrf",    "difference between BM25 and TF-IDF for document ranking"),
    ("mixed",     "pure_rrf",    "how does the mRNA COVID-19 vaccine trigger an immune response"),
    ("mixed",     "pure_rrf",    "is Rust actually memory safe compared to C++"),
    ("exact",     "sparse_only", "\"I have a dream\" which speech and what year"),
    ("exact",     "sparse_only", "lyrics never gonna give you up never gonna let you down"),
    ("rare-1tok", "sparse_only", "defenestration"),
    ("oov",       "sparse_only", "was ist die Hauptstadt von Osterreich"),
    ("code",      "sparse_only", "for i in range(len(arr)): arr[i] += 1"),
    ("common",    "dense_only",  "what is the best way to be happy in life"),
    ("common",    "dense_only",  "how can I get better at my job over time"),
]
_cats = pd.DataFrame(CASES, columns=["category", "expect", "query"])
_res = classify(_cats["query"].tolist())
_pcols = [c for c in _res.columns if c in ("dense_only", "sparse_only", "pure_rrf")]
_res.insert(0, "category", _cats["category"].to_numpy())
_res.insert(1, "expect", _cats["expect"].to_numpy())
_res["match"] = _res["route"] == _res["expect"]

# CONSTANT baselines vs the SAME heuristic: a sparse-default scores well because the battery is sparse-heavy.
router_match = _res["match"].mean()
const_match = {r: float((_cats["expect"] == r).mean()) for r in ("sparse_only", "dense_only", "pure_rrf")}
best_const_route = max(const_match, key=const_match.get)
print(f"router match vs heuristic: {router_match:.0%}")
print(f"constant baselines: " + ", ".join(f"always-{r} {v:.0%}" for r, v in const_match.items()))
print(f"router LIFT over best constant (always-{best_const_route} {const_match[best_const_route]:.0%}): "
      f"{router_match - const_match[best_const_route]:+.0%}   <- this is the real signal, not the raw match")
print(f"router served mix: {_res['route'].value_counts(normalize=True).round(2).to_dict()} "
      f"(one route dominating => constant-in-disguise)")
pd.set_option("display.max_colwidth", 62)
display(_res[["category", "query", "expect", "route", "match"] + _pcols].round(3))
print("\nbalanced per-category match (buried-id low = rarity blindness):")
display(_res.groupby("category")["match"].agg(["mean", "count"]).round(2))

router match vs heuristic: 36%
constant baselines: always-sparse_only 59%, always-dense_only 27%, always-pure_rrf 14%
router LIFT over best constant (always-sparse_only 59%): -23%   <- this is the real signal, not the raw match
router served mix: {'sparse_only': 0.45, 'pure_rrf': 0.32, 'dense_only': 0.23} (one route dominating => constant-in-disguise)


,category,query,expect,route,match,sparse_only,dense_only
0,id/code,CVE-2021-44228 log4j remote code execution,sparse_only,sparse_only,True,0.484,0.613
1,id/code,ORA-00942 table or view does not exist,sparse_only,sparse_only,True,0.388,0.621
2,id/code,kubectl pod CrashLoopBackOff exit code 137,sparse_only,sparse_only,True,0.575,0.683
3,id/code,pip ImportError libGL.so.1 cannot open shared object file,sparse_only,sparse_only,True,0.457,0.600
4,id/code,doi:10.1038/s41586-020-2649-2,sparse_only,pure_rrf,False,0.496,0.503
5,buried-id,why does my postgres connection fail with ECONNREFUSED 127...,sparse_only,pure_rrf,False,0.525,0.477
6,buried-id,what causes the E11000 duplicate key error in mongodb,sparse_only,pure_rrf,False,0.602,0.592
7,buried-id,how do I fix Objects are not valid as a React child,sparse_only,dense_only,False,0.399,0.763
8,concept,how does photosynthesis differ from cellular respiration,dense_only,pure_rrf,False,0.521,0.579
9,concept,what are the ethical implications of autonomous weapons,dense_only,sparse_only,False,0.475,0.582



balanced per-category match (buried-id low = rarity blindness):


,mean,count
category,,
buried-id,0.00,3
code,0.00,1
common,0.50,2
concept,0.25,4
exact,0.50,2
id/code,0.80,5
mixed,0.33,3
oov,0.00,1
rare-1tok,0.00,1


In [52]:
display(classify([
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j',
    'http://localhost.com',
    "why do cats purr"
]))

,sparse_only,dense_only,route,query
0,0.997652,0.004191,sparse_only,Who likes Curling?
1,0.380311,0.803069,dense_only,what are the side effects of DHA
2,0.357317,0.622248,sparse_only,CVE-2021-44228 log4j
3,0.397170,0.609269,sparse_only,http://localhost.com
4,0.419445,0.734358,sparse_only,why do cats purr


## 4e. Held-out evaluation — score ALL 6 saved arms (no refitting)

Loads every arm’s saved artifact and scores it on `FAIR_LANE`’s decisive cascade rows. Feature/LightGBM arms are not raw-query serve-safe, but their held-out rows can still assemble the saved feature/Zipf transforms. The check `meta.holdout_lane == FAIR_LANE` prevents leakage; the deployable comparison is the global constant from the training lanes.

In [ ]:
# 4e — score every SAVED arm on FAIR_LANE; compare to the per-lane ORACLE and the GLOBAL constant
RRF_DELTA = globals().get("RRF_DELTA", 0.15)
def _serve(probs, thr, delta=RRF_DELTA):        # tuned thresholds + pure_rrf hedge on near-ties
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); sp = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - sp) < delta, "pure_rrf", base)

def global_constant(table, holdout_lane):
    """DEPLOYABLE baseline: single best route over ALL training lanes (no lane label at serve time)."""
    tr = table.frame[(table.frame["dataset"] != holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    means = {r: tr[f"score_{r}"].mean() for r in ROUTES}
    return max(means, key=means.get)

def evaluate_saved_frame(arm_dir, table, holdout_lane, global_route):
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    assert meta.get("holdout_lane") == holdout_lane, (
        f"{p.name}: saved holdout={meta.get('holdout_lane')!r} != {holdout_lane!r} — retrain (leakage)")
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities

    frame = table.frame.reset_index(drop=True)
    emb = QueryEmbeddings(meta["embedding_model"]).matrix(frame)
    if meta.get("aux_embedding_model"):
        emb = np.concatenate([emb, QueryEmbeddings(meta["aux_embedding_model"]).matrix(frame)], axis=1)
    mask = ((frame["dataset"] == holdout_lane) & (frame["shape"] == "routes_differ")
            & frame["serve"].notna()).to_numpy()
    idx = np.flatnonzero(mask); q = frame.loc[idx, "query"]
    blocks = [emb[idx], svd.transform(q)]
    if meta["feature_inputs"]:
        f = table.feature_matrix.to_numpy(np.float32)[idx]
        blocks.append((f - np.load(p / "feat_mean.npy")) / np.load(p / "feat_std.npy"))
    if meta["zipf_inputs"]:
        z = ZipfStats().frame(q).to_numpy(np.float32)
        blocks.append((z - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    if meta.get("shape_inputs"):
        sh = LexicalShape().frame(q).to_numpy(np.float32)
        blocks.append((sh - np.load(p / "shape_mean.npy")) / np.load(p / "shape_std.npy"))
    probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
    served = _serve(probs, thr)

    tf = frame.loc[idx]; sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    lane_best = max(consts, key=consts.get)          # per-lane ORACLE (needs lane label -> NOT deployable)
    lane_c = consts[lane_best]; glob_c = consts[global_route]   # GLOBAL constant (deployable everywhere)
    orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    hr = lambda base: (cap - base) / (orc - base) if (orc - base) > 0.02 else np.nan
    return {"arm": p.name, "n_dec": len(idx), "captured": cap,
            "lane_route": lane_best, "lane_const": lane_c, "hr_vs_lane": hr(lane_c),
            "global_route": global_route, "global_const": glob_c,
            "lift_vs_global": cap - glob_c, "hr_vs_global": hr(glob_c),
            "oracle": orc, "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

GLOBAL_ROUTE = global_constant(table, FAIR_LANE)
print(f"held-out {FAIR_LANE} | GLOBAL constant (deployable) = {GLOBAL_ROUTE!r} | "
      f"per-lane oracle route may differ")
rows = []
for a in PROBE_ARMS:
    d = BASE / a.name
    if not (d / "meta.json").exists():
        print(f"[{a.name}] not saved yet — run §3b"); continue
    try:
        rows.append(evaluate_saved_frame(d, table, FAIR_LANE, GLOBAL_ROUTE))
    except AssertionError as e:
        print("SKIP:", e)
print("\nhr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);")
print("hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):")
display(pd.DataFrame(rows).round(3) if rows else "no saved models yet — run §3b")

held-out rarb-math | GLOBAL constant (deployable) = 'pure_rrf' | per-lane oracle route may differ

hr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);
hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):


,arm,n_dec,captured,lane_route,lane_const,hr_vs_lane,global_route,global_const,lift_vs_global,hr_vs_global,oracle,served_max
0,no_branches,4135,0.446,pure_rrf,0.493,-0.203,pure_rrf,0.493,-0.048,-0.203,0.728,0.665
1,shuffled_targets,4135,0.493,pure_rrf,0.493,0.000,pure_rrf,0.493,0.000,0.000,0.728,1.000
2,features_input_nocorpus,4135,0.437,pure_rrf,0.493,-0.237,pure_rrf,0.493,-0.056,-0.237,0.728,0.700
3,zipf_input_nocorpus,4135,0.439,pure_rrf,0.493,-0.230,pure_rrf,0.493,-0.054,-0.230,0.728,0.677
4,zipf_shape_nocorpus,4135,0.432,pure_rrf,0.493,-0.260,pure_rrf,0.493,-0.061,-0.260,0.728,0.532
5,bge_plus_e5_nocorpus,4135,0.453,pure_rrf,0.493,-0.172,pure_rrf,0.493,-0.040,-0.172,0.728,0.677
6,lightgbm_nocorpus,4135,0.455,pure_rrf,0.493,-0.164,pure_rrf,0.493,-0.038,-0.164,0.728,0.976
7,design_branches,4135,0.417,pure_rrf,0.493,-0.326,pure_rrf,0.493,-0.076,-0.326,0.728,0.524
8,bge_plus_gemini_nocorpus,4135,0.421,pure_rrf,0.493,-0.308,pure_rrf,0.493,-0.072,-0.308,0.728,0.531


## 4f. Zipf-rule router — the simplest thing (serve-safe, no model)

The learned arms can be compared with the constant (§4e), so state the mapping instead of learning it: a
3-threshold rule over `ZipfStats` (background word rarity). Rare/OOV tokens (CVE, log4j, error codes)
→ sparse; all-common words → dense; the ambiguous middle → the base constant. Deterministic,
interpretable, serve-safe (wordfreq bundled). Held-out headroom over the best constant is the same
bar as every learned arm — the curated demo will look great, so trust the §4f held-out number, not it.

In [ ]:
# 4f — pure Zipf-rule router: rare -> sparse, common -> dense, else base. No model. Serve-safe.
from encoder_router.table import ZipfStats

Z_RARE_SHARE  = 0.30    # >= this share of rare (zipf<3) tokens -> sparse (lexical / identifiers)
Z_OOV_SHARE   = 0.20    # >= this share of OOV tokens          -> sparse
Z_COMMON_MEAN = 4.50    # mean zipf >= this (all-common words) -> dense (semantic)
Z_BASE        = "sparse_only"   # ambiguous middle -> the per-lane constant

def load_classifier_with_zipf(rare=Z_RARE_SHARE, oov=Z_OOV_SHARE, common=Z_COMMON_MEAN, base=Z_BASE):
    zs = ZipfStats()
    def classify(queries):
        q = [queries] if isinstance(queries, str) else list(queries)
        zf = zs.frame(pd.Series(q))
        rs = zf["zipf.rare_share"].to_numpy(); ov = zf["zipf.oov_share"].to_numpy(); mn = zf["zipf.mean"].to_numpy()
        route = np.full(len(q), base, dtype=object)
        route = np.where(mn >= common, "dense_only", route)                 # all common -> dense
        route = np.where((rs >= rare) | (ov >= oov), "sparse_only", route)  # rare/oov -> sparse (wins)
        return pd.DataFrame({"query": q, "route": route}).join(zf.round(2))
    return classify

def evaluate_zipf_rule(table, holdout_lane, **kw):
    clf = load_classifier_with_zipf(**kw)
    tf = table.frame[(table.frame["dataset"] == holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    served = clf(tf["query"].tolist())["route"].to_numpy()
    sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    best = max(consts, key=consts.get); orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    return {"rule": "zipf", "n_decisive": len(tf), "best_route": best,
            "headroom": (cap - consts[best]) / (orc - consts[best]) if (orc - consts[best]) > 0.02 else np.nan,
            "raw_lift": cap - consts[best], "captured": cap, "best_const": consts[best], "oracle": orc,
            "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

zipf_router = load_classifier_with_zipf()
pd.set_option("display.max_colwidth", 55)
print("demo (looks great by construction — not the verdict):")
display(zipf_router([
    "CVE-2021-44228 log4j remote code execution", "http://localhost.com", "Who likes Curling?",
    "what are the side effects of DHA", "how does photosynthesis differ from cellular respiration",
]))
print(f"\nheld-out on {FAIR_LANE} (the real bar — beat best_const, served_max<0.95):")
display(pd.DataFrame([evaluate_zipf_rule(table, FAIR_LANE)]).round(3))

demo (looks great by construction — not the verdict):


,query,route,zipf.min,zipf.mean,zipf.max,zipf.rare_share,zipf.oov_share
0,CVE-2021-44228 log4j remote code execution,sparse_only,0.00,3.14,5.08,0.43,0.14
1,http://localhost.com,sparse_only,2.02,3.68,4.81,0.33,0.00
2,Who likes Curling?,dense_only,3.51,4.86,6.34,0.00,0.00
3,what are the side effects of DHA,dense_only,2.78,5.92,7.73,0.14,0.00
4,how does photosynthesis differ from cellular respir...,dense_only,3.03,4.66,6.63,0.00,0.00



held-out on rarb-math (the real bar — beat best_const, served_max<0.95):


,rule,n_decisive,best_route,headroom,raw_lift,captured,best_const,oracle,served_max
0,zipf,4135,pure_rrf,-0.183,-0.043,0.45,0.493,0.728,0.924


In [ ]:
display(zipf_router([
    "who loves cats?",
    "what is wrong with society",
    "OOOO #hash"
]))

,query,route,zipf.min,zipf.mean,zipf.max,zipf.rare_share,zipf.oov_share
0,who loves cats?,dense_only,4.46,5.15,6.34,0.0,0.0
1,what is wrong with society,dense_only,5.17,6.17,7.07,0.0,0.0
2,OOOO #hash,sparse_only,2.61,3.12,3.63,0.5,0.0


## 5. Balanced 45/45/10 mode — fair per-query routing test

Downsample the **decisive union** to equal-ish classes (rrf is the rarest, so it binds the
total size), dedup on normalized query text, then a stratified **90/10** split — both sides
carry 45/45/10. The constant baseline is therefore ~45%, and **per-route accuracy** is the
honest "can it route per-query?" bar the lane-holdout eval can't give. Balance is done by
*trimming*, not weights. Trains to an ISOLATED dir; the lane-holdout path (§3b/§4e) is untouched.


In [69]:
# 5a — whole dataset, one row per query, labelled by its TOP-SCORING route, trimmed to 45/45/10, split 90/10
import re as _re
BAL_MIX    = {"dense_only": 0.45, "sparse_only": 0.45, "pure_rrf": 0.10}
EVAL_SHARE = 0.10                                          # 90/10 split; both sides carry BAL_MIX
TRAIN_VERSION_BAL = f"{TRAIN_VERSION}-balanced-toproute-{sorted(BAL_MIX.items())}-e{EVAL_SHARE}"

def _norm_q(s):                                           # collapse case+whitespace so paraphrase dups don't leak across the split
    return _re.sub(r"\s+", " ", str(s).strip().lower())

def top_route(frame):
    """The route each query scores highest on — its route label. None where nothing scored > 0."""
    sc = frame[[f"score_{r}" for r in ROUTES]].to_numpy(float)
    return pd.Series(np.where(sc.max(1) > 0, np.array(ROUTES)[sc.argmax(1)], None), index=frame.index)

def balanced_split(union_frame, mix=BAL_MIX, eval_share=EVAL_SHARE, seed=SEED):
    """Whole dataset -> dedup on normalized query -> label by top-scoring route -> downsample to
    `mix` (the rarest route binds the total) -> stratified 90/10. Returns (frame, train_mask, eval_idx)."""
    f = union_frame.loc[~union_frame["query"].map(_norm_q).duplicated()].copy()
    f["route"] = top_route(f)
    f = f[f["route"].notna()]                                   # drop rows where no route surfaced anything
    counts = f["route"].value_counts()
    total = int(min(counts[c] / w for c, w in mix.items()))     # rarest route (rrf) sets the size
    rng = np.random.default_rng(seed)
    keep = np.concatenate([rng.choice(f.index[f["route"] == c].to_numpy(),
                                      int(round(total * w)), replace=False) for c, w in mix.items()])
    bal = f.loc[rng.permutation(keep)].reset_index(drop=True)
    eval_pos = np.concatenate([rng.choice(bal.index[bal["route"] == c].to_numpy(),
                                          int(round((bal["route"] == c).sum() * eval_share)), replace=False)
                               for c in mix])
    eval_mask = np.zeros(len(bal), bool); eval_mask[eval_pos] = True
    return bal, ~eval_mask, np.flatnonzero(eval_mask)

_bal_src = AcceptabilityLabels(union, tolerance=None).frame().reset_index(drop=True)   # union already strips __-suffixed score cols
bal, TRAIN_MASK, EVAL_IDX = balanced_split(_bal_src)
bal_table = TrainingTable(); bal_table.__dict__["frame"] = bal
print(f"balanced pop {len(bal):,} of {len(_bal_src):,} rows | route mix {bal['route'].value_counts(normalize=True).round(3).to_dict()}")
print(f"train {int(TRAIN_MASK.sum()):,} | eval {len(EVAL_IDX):,} "
      f"| eval mix {bal.loc[EVAL_IDX, 'route'].value_counts(normalize=True).round(3).to_dict()}")


balanced pop 73,440 of 233,247 rows | route mix {'sparse_only': 0.45, 'dense_only': 0.45, 'pure_rrf': 0.1}
train 66,096 | eval 7,344 | eval mix {'sparse_only': 0.45, 'dense_only': 0.45, 'pure_rrf': 0.1}


In [70]:
# 5b — train each arm on the balanced 90% (held-out 10% excluded via TRAIN_MASK); ISOLATED dir
BASE_BAL = MODELS_DIR / "os_distill_relabel_balanced"

def _bal_cache_ok(arm):
    """Reusable only if the saved manifest matches this run's balanced config (else stale/leaked)."""
    mp = BASE_BAL / arm.name / "meta.json"
    if not mp.exists():
        return False
    try:
        m = _json.loads(mp.read_text())
    except Exception:
        return False
    weights = "lgbm.joblib" if arm.learner == "lgbm" else "router.pt"
    return (m.get("train_version") == TRAIN_VERSION_BAL and m.get("split_mode") == "balanced"
            and (BASE_BAL / arm.name / weights).exists())

_todo_b = list(PROBE_ARMS) if FORCE_RETRAIN else [a for a in PROBE_ARMS if not _bal_cache_ok(a)]
print(f"balanced training | {len(bal):,} rows | to train: {[a.name for a in _todo_b] or '-'}", flush=True)
for _n, _a in enumerate(_todo_b, 1):
    print(f"===== balanced arm {_n}/{len(_todo_b)}: {_a.name} =====", flush=True)
    train_arm(bal_table, _a, BASE_BAL, pool_mask=TRAIN_MASK,
              train_version=TRAIN_VERSION_BAL, split_mode="balanced")
    print(flush=True)
print("balanced arms saved." if _todo_b else "nothing to train - all balanced arms cached.")


balanced training | 73,440 rows | to train: ['no_branches', 'shuffled_targets', 'features_input_nocorpus', 'zipf_input_nocorpus', 'zipf_shape_nocorpus', 'bge_plus_e5_nocorpus', 'bge_plus_e5_zipf', 'lightgbm_nocorpus', 'design_branches']
===== balanced arm 1/9: no_branches =====
  [no_branches] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [no_branches] +   0.0s  holdout_lane=None | train on 66,096/73,440 rows (44 lanes)
  [no_branches] +   2.4s  embeddings (73440, 384)
  [no_branches] +  13.0s  ngram-SVD fit (train lanes only)
  [no_branches] +  16.8s  input matrix x=(73440, 512)
  [no_branches] +  16.8s  branches: cell=None corpus=None feature=None
  [no_branches] +  16.9s  mlp fit: 59,487 train / 6,609 val  (fit bar tracks epochs)


fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [no_branches] +  21.2s  mlp done: 24 epochs, best_val=0.2423
  [no_branches] +  21.2s  router.pt saved
  [no_branches] +  21.4s  tuned thresholds [0.3, 0.85]
  [no_branches] +  21.4s  served mix (train lanes): {'sparse_only': 0.87, 'dense_only': 0.13}
  [no_branches] +  21.5s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/os_distill_relabel_balanced/no_branches  (held out None, total 21.5s)

===== balanced arm 2/9: shuffled_targets =====
  [shuffled_targets] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [shuffled_targets] +   0.0s  holdout_lane=None | train on 66,096/73,440 rows (44 lanes)
  [shuffled_targets] +   2.5s  embeddings (73440, 384)
  [shuffled_targets] +  12.6s  ngram-SVD fit (train lanes only)
  [shuffled_targets] +  16.4s  input matrix x=(73440, 512)
extracting features for 38148 unindexed queries
  [shuffled_targets] +  66.3s  branches: cell=(73440, 44) corpus=None feature=None
  [shuffled_targets] +  66.3s  shuffled ROUTE labe

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [shuffled_targets] +  73.8s  mlp done: 26 epochs, best_val=0.2654
  [shuffled_targets] +  73.8s  router.pt saved
  [shuffled_targets] +  74.0s  tuned thresholds [0.3, 0.55]
  [shuffled_targets] +  74.0s  served mix (train lanes): {'sparse_only': 1.0}
  [shuffled_targets] +  74.0s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/os_distill_relabel_balanced/shuffled_targets  (held out None, total 74.0s)

===== balanced arm 3/9: features_input_nocorpus =====
  [features_input_nocorpus] +   0.0s  START learner=mlp feature_inputs=True zipf_inputs=False
  [features_input_nocorpus] +   0.0s  holdout_lane=None | train on 66,096/73,440 rows (44 lanes)
  [features_input_nocorpus] +   2.4s  embeddings (73440, 384)
  [features_input_nocorpus] +  12.8s  ngram-SVD fit (train lanes only)
  [features_input_nocorpus] +  16.7s  assembling taxonomy feature inputs...
  [features_input_nocorpus] +  16.8s  feature inputs (73440, 83)
  [features_input_nocorpus] +  16.8s  input matrix x=(73

...

  [bge_plus_e5_nocorpus] +  26.3s  mlp done: 25 epochs, best_val=0.2415
  [bge_plus_e5_nocorpus] +  26.3s  router.pt saved
  [bge_plus_e5_nocorpus] +  26.5s  tuned thresholds [0.3, 0.85]
  [bge_plus_e5_nocorpus] +  26.5s  served mix (train lanes): {'sparse_only': 0.88, 'dense_only': 0.12}
  [bge_plus_e5_nocorpus] +  26.5s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/os_distill_relabel_balanced/bge_plus_e5_nocorpus  (held out None, total 26.5s)

===== balanced arm 7/9: bge_plus_e5_zipf =====
  [bge_plus_e5_zipf] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=True
  [bge_plus_e5_zipf] +   0.0s  holdout_lane=None | train on 66,096/73,440 rows (44 lanes)
  [bge_plus_e5_zipf] +   2.3s  embeddings (73440, 384)
  [bge_plus_e5_zipf] +   4.3s  aux encoder intfloat/multilingual-e5-small -> (73440, 768)
  [bge_plus_e5_zipf] +  14.4s  ngram-SVD fit (train lanes only)
  [bge_plus_e5_zipf] +  19.3s  zipf inputs (73440, 5)
  [bge_plus_e5_zipf] +  20.4s  shape inputs

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [bge_plus_e5_zipf] +  26.3s  mlp done: 27 epochs, best_val=0.2408
  [bge_plus_e5_zipf] +  26.3s  router.pt saved
  [bge_plus_e5_zipf] +  26.4s  tuned thresholds [0.3, 0.9]
  [bge_plus_e5_zipf] +  26.4s  served mix (train lanes): {'sparse_only': 0.89, 'dense_only': 0.11}
  [bge_plus_e5_zipf] +  26.4s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/os_distill_relabel_balanced/bge_plus_e5_zipf  (held out None, total 26.4s)

===== balanced arm 8/9: lightgbm_nocorpus =====
  [lightgbm_nocorpus] +   0.0s  START learner=lgbm feature_inputs=True zipf_inputs=False
  [lightgbm_nocorpus] +   0.0s  holdout_lane=None | train on 66,096/73,440 rows (44 lanes)
  [lightgbm_nocorpus] +   2.5s  embeddings (73440, 384)
  [lightgbm_nocorpus] +  13.2s  ngram-SVD fit (train lanes only)
  [lightgbm_nocorpus] +  17.4s  assembling taxonomy feature inputs...
  [lightgbm_nocorpus] +  17.5s  feature inputs (73440, 83)
  [lightgbm_nocorpus] +  17.5s  input matrix x=(73440, 595)
  [lightgbm_nocor

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [design_branches] +  28.5s  mlp done: 36 epochs, best_val=0.2405
  [design_branches] +  28.5s  router.pt saved
  [design_branches] +  28.7s  tuned thresholds [0.3, 0.85]
  [design_branches] +  28.7s  served mix (train lanes): {'sparse_only': 0.9, 'dense_only': 0.1}
  [design_branches] +  28.7s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/os_distill_relabel_balanced/design_branches  (held out None, total 28.7s)

balanced arms saved.


In [72]:
# 5c — score each saved balanced arm on the held-out 10%: per-route accuracy vs the ~45% constant
RRF_DELTA = globals().get("RRF_DELTA", 0.07)
def _serve_bal(probs, thr, delta=RRF_DELTA):              # tuned thresholds + pure_rrf hedge on near-ties (same rule as §4e)
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); sp = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - sp) < delta, "pure_rrf", base)

def evaluate_balanced(arm_dir, table, eval_idx):
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    assert meta.get("split_mode") == "balanced", f"{p.name}: not a balanced model — run §5b"
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities
    frame = table.frame
    emb = QueryEmbeddings(meta["embedding_model"]).matrix(frame)
    if meta.get("aux_embedding_model"):
        emb = np.concatenate([emb, QueryEmbeddings(meta["aux_embedding_model"]).matrix(frame)], axis=1)
    q = frame.loc[eval_idx, "query"]
    blocks = [emb[eval_idx], svd.transform(q)]
    if meta["feature_inputs"]:                             # taxonomy feature block (order matches train_arm: after svd, before zipf)
        f = table.feature_matrix.to_numpy(np.float32)[eval_idx]
        blocks.append((f - np.load(p / "feat_mean.npy")) / np.load(p / "feat_std.npy"))
    if meta["zipf_inputs"]:
        blocks.append((ZipfStats().frame(q).to_numpy(np.float32) - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    if meta.get("shape_inputs"):
        blocks.append((LexicalShape().frame(q).to_numpy(np.float32) - np.load(p / "shape_mean.npy")) / np.load(p / "shape_std.npy"))
    probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
    served = _serve_bal(probs, thr)
    ev = frame.loc[eval_idx]; true = ev["route"].to_numpy()   # the route it scores highest on
    sc = {r: ev[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = float(np.array([sc[r][i] for i, r in enumerate(served)]).mean())
    orc = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
    per = {f"acc_{r.split('_')[0]}": (float((served[true == r] == r).mean()) if (true == r).any() else np.nan)
           for r in ROUTES}
    const = float(pd.Series(true).value_counts(normalize=True).max())   # best single-route accuracy on the balanced eval (~0.45)
    return {"arm": p.name, "n_eval": len(eval_idx), "acc": float((served == true).mean()),
            "const_acc": const, "lift_vs_const": float((served == true).mean()) - const, **per,
            "captured": cap, "oracle": orc,
            "served_mix": pd.Series(served).value_counts(normalize=True).round(2).to_dict()}

rows = []
for a in PROBE_ARMS:
    d = BASE_BAL / a.name
    if not (d / "meta.json").exists():
        print(f"[{a.name}] not saved — run §5b"); continue
    try:
        rows.append(evaluate_balanced(d, bal_table, EVAL_IDX))
    except AssertionError as e:
        print("SKIP:", e)
print("acc = per-query top-route accuracy on the balanced held-out; const_acc ~0.45 is the bar to beat;")
print("acc_dense/pure/sparse = per-true-class recall (does it pick the minority routes, not just sparse):")
display(pd.DataFrame(rows).round(3) if rows else "no balanced models — run §5b")


acc = per-query top-route accuracy on the balanced held-out; const_acc ~0.45 is the bar to beat;
acc_dense/pure/sparse = per-true-class recall (does it pick the minority routes, not just sparse):


,arm,n_eval,acc,const_acc,lift_vs_const,acc_dense,acc_pure,acc_sparse,captured,oracle,served_mix
0,no_branches,7344,0.448,0.45,-0.002,0.195,0.151,0.767,0.580,0.681,"{'sparse_only': 0.71, 'pure_rrf': 0.15, 'dense_only': 0.13}"
1,shuffled_targets,7344,0.100,0.45,-0.350,0.000,1.000,0.000,0.491,0.681,{'pure_rrf': 1.0}
2,features_input_nocorpus,7344,0.430,0.45,-0.020,0.107,0.162,0.813,0.575,0.681,"{'sparse_only': 0.78, 'pure_rrf': 0.15, 'dense_only': 0.07}"
3,zipf_input_nocorpus,7344,0.428,0.45,-0.022,0.128,0.165,0.787,0.576,0.681,"{'sparse_only': 0.75, 'pure_rrf': 0.16, 'dense_only': 0.09}"
4,zipf_shape_nocorpus,7344,0.460,0.45,0.010,0.202,0.147,0.786,0.582,0.681,"{'sparse_only': 0.73, 'pure_rrf': 0.14, 'dense_only': 0.13}"
5,bge_plus_e5_nocorpus,7344,0.435,0.45,-0.015,0.170,0.161,0.761,0.575,0.681,"{'sparse_only': 0.72, 'pure_rrf': 0.16, 'dense_only': 0.12}"
6,bge_plus_e5_zipf,7344,0.446,0.45,-0.004,0.169,0.153,0.788,0.580,0.681,"{'sparse_only': 0.74, 'pure_rrf': 0.15, 'dense_only': 0.11}"
7,lightgbm_nocorpus,7344,0.445,0.45,-0.005,0.153,0.185,0.794,0.592,0.681,"{'sparse_only': 0.7, 'pure_rrf': 0.2, 'dense_only': 0.1}"
8,design_branches,7344,0.452,0.45,0.001,0.165,0.144,0.806,0.585,0.681,"{'sparse_only': 0.76, 'pure_rrf': 0.13, 'dense_only': 0.11}"


### 5d. Eye test — classify free-text queries with a BALANCED arm

Loads one balanced arm (serve-safe arms only) and routes whatever you type. Serving is the
same as §4e: tuned thresholds + a pure_rrf hedge on near-ties. Reuses `load_classifier`
from §4c — run §4c once first if `load_classifier` isn't defined yet.


In [77]:
# 5d — eye test a balanced arm. Swap the arm name; feature_inputs arms are NOT serve-safe (skip them).
bal_classify = load_classifier(BASE_BAL / "zipf_shape_nocorpus")     # e.g. "zipf_shape_nocorpus", "bge_plus_e5_nocorpus"
pd.set_option("display.max_colwidth", 70)
_res = bal_classify([
    "how do I refresh laravel migrations programmatically",
    "United States v. Fumo evidence Pennsylvania Ethics Act admissibility",
    "what is the difference between computer engineering and computer science",
    "CVE-2021-44228 log4j mitigation steps",
    "localhost:8080 connection refused",
    "best pizza near me open now",
])
_res["argmax"] = _res[list(HEAD_ROUTES)].idxmax(axis=1)      # model's raw pick (higher head prob), ignoring the cost thresholds
display(_res[[*HEAD_ROUTES, "argmax", "route", "query"]])    # argmax != route -> the tuned threshold flipped it (usually to cheaper sparse)


,sparse_only,dense_only,argmax,route,query
0,0.447392,0.672161,dense_only,sparse_only,how do I refresh laravel migrations programmatically
1,0.721700,0.481415,sparse_only,sparse_only,United States v. Fumo evidence Pennsylvania Ethics Act admissibility
2,0.284621,0.749950,dense_only,dense_only,what is the difference between computer engineering and computer s...
3,0.411777,0.550572,dense_only,sparse_only,CVE-2021-44228 log4j mitigation steps
4,0.463445,0.576889,dense_only,sparse_only,localhost:8080 connection refused
5,0.209523,0.757897,dense_only,dense_only,best pizza near me open now


In [78]:
display(bal_classify([
    "CVE-2021-44228 log4j",
    "why do cats purr",
    "I have a cat",
    "What is qdrant?", 
    "I love doing this. Ohoho", 
    "Where is my heart",
    "Sheakspeare is great writer",
    "https://chatgpt.com/",
    "Aloha",
    "site:qdrant.tech documentation quickstart docker qdrant official"
]))


,sparse_only,dense_only,route,query
0,0.418027,0.483779,pure_rrf,CVE-2021-44228 log4j
1,0.290629,0.735437,dense_only,why do cats purr
2,0.219001,0.814000,dense_only,I have a cat
3,0.530474,0.589799,pure_rrf,What is qdrant?
4,0.575437,0.380228,sparse_only,I love doing this. Ohoho
5,0.251196,0.776607,dense_only,Where is my heart
6,0.445747,0.648482,sparse_only,Sheakspeare is great writer
7,0.315874,0.427355,sparse_only,https://chatgpt.com/
8,0.712743,0.632278,sparse_only,Aloha
9,0.498763,0.644320,sparse_only,site:qdrant.tech documentation quickstart docker qdrant official
